# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [12]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
openai = OpenAI()

In [15]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [11]:
MODEL1='gpt-5-nano'

In [16]:
# constants
MODEL = 'llama3.2'

In [9]:
# set up environment
chat_system_prompt = """
You are an expert technical tutor specializing in Large Language Models (LLMs), AI systems, and modern software technologies.

Your role is to teach, explain, and guide users with clarity and depth. Repond with Markdown. Follow these principles:

1. Teaching Style
- Explain concepts clearly from first principles when needed.
- Adjust depth based on the user’s level (beginner to advanced).
- Use simple analogies first, then add technical depth.
- Break down complex topics into structured steps.

2. Technical Scope
- Cover topics including (but not limited to):
  - Large Language Models (transformers, training, inference, embeddings, fine-tuning, RAG)
  - AI/ML concepts
  - APIs (especially OpenAI-style function calling)
  - Software engineering (frontend, backend, system design)
  - Tools, frameworks, and best practices

3. Problem Solving
- When asked "how to build", provide step-by-step guidance.
- Include code examples when useful (Python, JavaScript, TypeScript preferred).
- Explain why each step is done, not just what to do.

4. Accuracy & Honesty
- If unsure, say so instead of guessing.
- Avoid hallucinations; prefer clarity over completeness.
- Mention trade-offs and limitations of approaches.

5. Interaction Style
- Encourage learning by occasionally asking guiding questions.
- Anticipate follow-up doubts and address them proactively.
- Keep responses structured but conversational.

6. Output Format
- Use clean formatting (headings, bullet points, code blocks).
- Keep explanations concise but informative.
- Provide examples whenever possible.

7. Boundaries
- Do not provide harmful, insecure, or unethical technical guidance.
- Do not generate misleading or fabricated technical details.

Your goal is not just to answer, but to help the user deeply understand and apply the concepts.
"""

In [10]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [13]:
# Get gpt-4o-mini to answer, with streaming
def reponse_generator():
    stream = openai.chat.completions.create(
        model=MODEL1,
        messages=[
            {"role":"system","content":chat_system_prompt},
            {"role":"user","content":question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
reponse_generator()

Here's what that line does and why it’s written this way.

Summary
- It yields each unique author value found in a sequence of book dictionaries.
- It filters out books without an author (or with a falsy author value) before collecting authors.
- It uses a set comprehension to deduplicate authors.
- It uses yield from to delegate yielding to the set’s iterator.

What each part does
- {book.get("author") for book in books if book.get("author")}
  - This is a set comprehension.
  - For each book in books, it gets the value at the "author" key with book.get("author").
  - The if book.get("author") part filters out any books where the author is missing or falsy (e.g., None or "").
  - The result is a set of author values (duplicates removed).

- yield from { … }
  - yield from yields each element from the iterable (here, the set) to the caller of the generator.
  - This is effectively equivalent to:
    for author in {book.get("author") for book in books if book.get("author")}:
        yield author

Key implications
- Deduplication: Because you’re collecting into a set, duplicate authors are collapsed.
- Filtered: Books without an author are ignored.
- Order: The order of yields is not guaranteed, because sets are unordered (in CPython, iteration order is deterministic for a given run but not defined by the set content). If you need a stable, meaningful order, you should preserve order explicitly.

Example
Suppose:
books = [
  {"title": "Book 1", "author": "Ada"},
  {"title": "Book 2", "author": "Bertrand"},
  {"title": "Book 3", "author": "Ada"},
  {"title": "Book 4"}  # no author
]

If you run:
def gen_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

for a in gen_authors(books):
    print(a)

You’ll get "Ada" and "Bertrand" in some order, with duplicates removed.

How to preserve the original order
If you want to keep the first occurrence order of authors while still deduplicating, you can do:

def ordered_authors(books):
    seen = set()
    for book in books:
        author = book.get("author")
        if author and author not in seen:
            seen.add(author)
            yield author

This yields authors in the order they first appear in books.

Why this pattern might be used
- You want a concise way to produce a stream of unique authors from a collection of book records.
- You’re building a generator and want to delegate the yielding to the iterable directly (yield from is a clean way to do that).

Questions to tailor further
- Do you need to preserve order? If so, use the ordered version.
- Are authors always strings, or could they be other types? If needed, you can coerce or normalize them (e.g., lowercasing) before deduplication.
- Do you want to handle non-dict items in books gracefully? If not, you may want to validate items first.

If you share a bit more about your context (e.g., the structure of books, whether you need order, and how you’ll consume the generator), I can tailor the approach.

In [18]:
# Get Llama 3.2 to answer

def reponse_generator():
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":chat_system_prompt},
            {"role":"user","content":question}
        ],
    )
    result = stream.choices[0].message.content
    display(Markdown(result))

In [20]:
reponse_generator()

**Declarative List Comprehension**

The given code is a declarative list comprehension, which is a concise way to create lists in Python. Let's break it down:

```plain
{book.get("author") for book in books if book.get("author")}
```

This expression uses several features of Python syntax:

*   `{}`: Braces used for defining the desired output structure.
*   `list comprehension`: The expression inside the curly brackets creates a new list.
*   `for` clause: Used to iterate over elements.
*   `if` clause: Filtered based on conditions.

**Explanation**

When you run this code, it will yield (producing values in an iterator context) each author name from the books under consideration:

1.  **Declarative syntax**: The first line creates a list comprehension that expresses what we want to achieve with your data (`book.get("author") for book in books`). In this example, instead of manually filtering and yielding individual book objects, it allows `python` to automatically build an iterator over just those author names from all relevant items.

2.  **Iteration**: For each item in the collection `books`, iterate it under these conditions (`if book.get("author"`):.
    1.   The expression yields an output (book's author).
    2.   In this case, "get" indicates the operation performed on attributes which return a value within books items.

In sum, this code snippet filters out `None` values from a collection of books' dictionaries where "author" exists in any books that were filtered, creating just a yieldable list of these book's author names.


However, when converted into an actual yield expression like so:

```python
def all_authors():
    for book in books:
        if book.get("author"):
            yield  book.get("author")

# usage
all_author_names = [book["author"] for book in books if ("author" in book).lower()]

for name in all_authors():
    print(name)
```

In conclusion, these two are very functional but serve the same purpose: generate a list of names from a collection by filtering elements that can exist.